# Exercise 3 — Attribute Exploration

Redefined task (see `CLAUDE.md` §3 Exercise 3): choose an attribute set from
a comfortable domain, run attribute exploration, collect/simulate expert
knowledge, and report what was learned. **Domain: wine chemistry**, kept
consistent with Exercises 1, 2 and 4 rather than switching to an unrelated
domain.

**Attributes (12)**, single-threshold booleans (deliberately *not* the
multi-level ordinal scales from Exercise 2 — using independent yes/no
attributes here keeps the canonical base from being dominated by trivial
same-attribute ordinal chains, which is exactly the kind of scale artifact
Exercise 2 already documented):

| Attribute | Definition |
|---|---|
| `high_fixed_acidity` | fixed acidity >= 8 g/L |
| `high_volatile_acidity` | volatile acidity >= 0.5 g/L |
| `citric_acid_present` | citric acid >= 0.25 g/L |
| `high_residual_sugar` | residual sugar >= 4 g/L |
| `high_chlorides` | chlorides >= 0.08 g/L |
| `high_free_so2` | free SO2 >= 15 mg/L |
| `high_total_so2` | total SO2 >= 60 mg/L |
| `high_density` | density >= 0.997 g/cm3 |
| `high_pH` | pH >= 3.4 |
| `high_sulphates` | sulphates >= 0.6 g/L |
| `high_alcohol` | alcohol >= 11% |
| `good_quality` | quality >= 7 |

**Expert oracle**: rather than relying on subjective domain claims, the
"expert" answering each proposed implication is the **full 1,599-row real
dataset** (`data/winequality-red.csv`) — every confirm/refute decision is a
real lookup against actual lab-measured wine data, and every counterexample
is a real wine. This is a stronger and more defensible form of "collecting
relevant knowledge from experts in the field" than hand-curated judgment
calls would be.

In [1]:
import pandas as pd
import itertools
from collections import Counter

df = pd.read_csv("../data/winequality-red.csv", sep=";")
df.columns = [c.strip() for c in df.columns]

ATTRS = ["high_fixed_acidity", "high_volatile_acidity", "citric_acid_present", "high_residual_sugar",
         "high_chlorides", "high_free_so2", "high_total_so2", "high_density", "high_pH",
         "high_sulphates", "high_alcohol", "good_quality"]


def row_attrs(row):
    a = set()
    if row["fixed acidity"] >= 8: a.add("high_fixed_acidity")
    if row["volatile acidity"] >= 0.5: a.add("high_volatile_acidity")
    if row["citric acid"] >= 0.25: a.add("citric_acid_present")
    if row["residual sugar"] >= 4: a.add("high_residual_sugar")
    if row["chlorides"] >= 0.08: a.add("high_chlorides")
    if row["free sulfur dioxide"] >= 15: a.add("high_free_so2")
    if row["total sulfur dioxide"] >= 60: a.add("high_total_so2")
    if row["density"] >= 0.997: a.add("high_density")
    if row["pH"] >= 3.4: a.add("high_pH")
    if row["sulphates"] >= 0.6: a.add("high_sulphates")
    if row["alcohol"] >= 11: a.add("high_alcohol")
    if row["quality"] >= 7: a.add("good_quality")
    return a


ORACLE = {idx: row_attrs(row) for idx, row in df.iterrows()}
print(f"{len(ORACLE)} wines in the oracle (full dataset)")

cnt = Counter()
for a in ORACLE.values():
    cnt.update(a)
for attr in ATTRS:
    print(f"  {attr}: {cnt[attr]}/{len(ORACLE)} ({cnt[attr]/len(ORACLE):.0%})")


1599 wines in the oracle (full dataset)
  high_fixed_acidity: 783/1599 (49%)
  high_volatile_acidity: 886/1599 (55%)
  citric_acid_present: 832/1599 (52%)
  high_residual_sugar: 136/1599 (9%)
  high_chlorides: 791/1599 (49%)
  high_free_so2: 753/1599 (47%)
  high_total_so2: 434/1599 (27%)
  high_density: 702/1599 (44%)
  high_pH: 424/1599 (27%)
  high_sulphates: 945/1599 (59%)
  high_alcohol: 467/1599 (29%)
  good_quality: 217/1599 (14%)


## Exploration algorithm

Same approach validated in earlier exercises: rather than a hand-rolled
lectic-order NextClosure (which produced subtle bugs when first attempted —
see the conversation history), this uses the provably-equivalent
brute-force-by-increasing-subset-size method. With only 12 attributes,
4,096 subsets is trivial to scan, so there's no efficiency reason to risk
the more error-prone lectic stepping logic.

**Why this is equivalent to textbook NextClosure**: a pseudo-intent's
defining condition only depends on implications whose premise is a *proper
subset* of it, and for finite sets, proper-subset always means strictly
fewer elements. So processing all subsets in increasing-size order is a
valid linear extension of the inclusion order — sufficient for correctness,
just less efficient than lectic order for large attribute sets (irrelevant
here).

In [2]:
def oracle_check(premise, conclusion):
    """Return None if the implication holds for every wine, else a counterexample wine id."""
    for wid, attrs in ORACLE.items():
        if premise <= attrs and not (conclusion <= attrs):
            return wid
    return None


def lin_L_closure(P, L):
    P = set(P)
    changed = True
    while changed:
        changed = False
        for prem, conc in L:
            if prem <= P and not (conc <= P):
                P |= conc
                changed = True
    return P


def context_closure(P, context_objs):
    if not context_objs:
        return set(ATTRS)  # vacuous: empty context -> intension = everything
    extent = [wid for wid in context_objs if P <= ORACLE[wid]]
    if not extent:
        return set(ATTRS)
    inter = set(ATTRS)
    for wid in extent:
        inter &= ORACLE[wid]
    return inter


all_subsets = [frozenset(c) for size in range(len(ATTRS) + 1) for c in itertools.combinations(ATTRS, size)]

L = []
context_objs = []
transcript = []
step = 0
while True:
    step += 1
    found = None
    for P in all_subsets:
        if lin_L_closure(P, L) != P:
            continue
        cc = context_closure(P, context_objs)
        if cc != P:
            found = (P, cc)
            break
    if found is None:
        break
    P, cc = found
    conclusion = cc - P
    violator = oracle_check(P, conclusion)
    if violator is None:
        L.append((frozenset(P), frozenset(conclusion)))
        transcript.append(("CONFIRM", P, conclusion, None))
    else:
        if violator not in context_objs:
            context_objs.append(violator)
        transcript.append(("REFUTE", P, conclusion, violator))

print(f"Terminated after {step} steps")
print(f"{len(L)} implications in the canonical base")
print(f"{len(context_objs)} distinct counterexample wines used")
confirms = sum(1 for t in transcript if t[0] == "CONFIRM")
refutes = sum(1 for t in transcript if t[0] == "REFUTE")
print(f"{confirms} confirmed, {refutes} refuted")


Terminated after 282 steps
123 implications in the canonical base
158 distinct counterexample wines used
123 confirmed, 158 refuted


In [3]:
# Soundness check: re-verify every accepted implication against the FULL oracle
violations = 0
for prem, conc in L:
    for wid, attrs in ORACLE.items():
        if prem <= attrs and not (conc <= attrs):
            violations += 1
print(f"{violations} violations found across the full 1,599-wine oracle (expect 0)")


0 violations found across the full 1,599-wine oracle (expect 0)


## First 20 steps of the exploration session

Shows the algorithm starting from the empty premise (proposing "every wine
has all 12 attributes," immediately refuted) and progressively narrowing.

In [4]:
for i, (kind, P, Q, v) in enumerate(transcript[:20], 1):
    tag = f"  [counterexample: wine #{v}]" if v is not None else ""
    print(f"{i}. {kind}: {sorted(P)} -> {sorted(Q)}{tag}")


1. REFUTE: [] -> ['citric_acid_present', 'good_quality', 'high_alcohol', 'high_chlorides', 'high_density', 'high_fixed_acidity', 'high_free_so2', 'high_pH', 'high_residual_sugar', 'high_sulphates', 'high_total_so2', 'high_volatile_acidity']  [counterexample: wine #0]
2. REFUTE: [] -> ['high_density', 'high_pH', 'high_volatile_acidity']  [counterexample: wine #1]
3. REFUTE: [] -> ['high_volatile_acidity']  [counterexample: wine #3]
4. REFUTE: ['high_fixed_acidity'] -> ['citric_acid_present', 'high_density', 'high_free_so2', 'high_total_so2']  [counterexample: wine #14]
5. REFUTE: ['high_fixed_acidity'] -> ['high_density', 'high_free_so2', 'high_total_so2']  [counterexample: wine #16]
6. REFUTE: ['high_fixed_acidity'] -> ['high_free_so2', 'high_total_so2']  [counterexample: wine #17]
7. REFUTE: ['high_fixed_acidity'] -> ['high_free_so2']  [counterexample: wine #23]
8. REFUTE: ['citric_acid_present'] -> ['high_fixed_acidity', 'high_free_so2']  [counterexample: wine #9]
9. REFUTE: ['citric

## Premise-size distribution of the canonical base

A first, striking result: **no implication with premise size 1 or 2 survives**
— every single-attribute and every pair-attribute candidate was refuted by
some real wine. The smallest exception-free implications need 3 simultaneous
conditions.

In [5]:
sizes = Counter(len(p) for p, q in L)
for s in sorted(sizes):
    print(f"premise size {s}: {sizes[s]} implications")


premise size 3: 29 implications
premise size 4: 39 implications
premise size 5: 40 implications
premise size 6: 8 implications
premise size 7: 2 implications
premise size 8: 2 implications
premise size 9: 2 implications
premise size 11: 1 implications


## The 29 size-3 implications (the simplest ones in the base)

In [6]:
size3 = [(p, q) for p, q in L if len(p) == 3]
for p, q in size3:
    print(sorted(p), "->", sorted(q))


['good_quality', 'high_fixed_acidity', 'high_volatile_acidity'] -> ['high_sulphates']
['high_fixed_acidity', 'high_residual_sugar', 'high_total_so2'] -> ['high_density']
['high_fixed_acidity', 'high_pH', 'high_residual_sugar'] -> ['citric_acid_present', 'high_alcohol', 'high_sulphates', 'high_volatile_acidity']
['high_alcohol', 'high_fixed_acidity', 'high_residual_sugar'] -> ['citric_acid_present']
['good_quality', 'high_fixed_acidity', 'high_residual_sugar'] -> ['citric_acid_present']
['good_quality', 'high_fixed_acidity', 'high_free_so2'] -> ['high_sulphates']
['good_quality', 'high_fixed_acidity', 'high_total_so2'] -> ['citric_acid_present', 'high_free_so2', 'high_sulphates']
['good_quality', 'high_fixed_acidity', 'high_pH'] -> ['high_sulphates']
['good_quality', 'high_alcohol', 'high_fixed_acidity'] -> ['citric_acid_present']
['citric_acid_present', 'good_quality', 'high_volatile_acidity'] -> ['high_sulphates']
['good_quality', 'high_total_so2', 'high_volatile_acidity'] -> ['high_a

In [7]:
# Save the full transcript and canonical base for reference
import csv

with open("../data/ex3_transcript.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "decision", "premise", "conclusion", "counterexample_wine_id"])
    for i, (kind, P, Q, v) in enumerate(transcript, 1):
        writer.writerow([i, kind, ";".join(sorted(P)), ";".join(sorted(Q)), v if v is not None else ""])

with open("../data/ex3_canonical_base.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["premise", "conclusion", "premise_size"])
    for p, q in L:
        writer.writerow([";".join(sorted(p)), ";".join(sorted(q)), len(p)])

print("Saved data/ex3_transcript.csv and data/ex3_canonical_base.csv")


Saved data/ex3_transcript.csv and data/ex3_canonical_base.csv


## A few real counterexample wines

Showing the actual measured values behind three of the 158 counterexamples,
to make clear these are real lab data, not synthetic.

In [8]:
sample_counterexamples = [t[3] for t in transcript if t[0] == "REFUTE"][:3]
df.loc[sample_counterexamples]


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
